# 🧪 PT-W2-D6 概念实验：Policy 抽取 + Ontology Compiler 初探

> 配套阅读：`PT-W2-D6-Policy抽取-OntologyCompiler初探.md`
> 把审批流变成 Policy 声明，模拟 Ontology Compiler 编译为 Design Artifact。

## 第 1 格：Policy 声明——审批流的三要素

In [ ]:
from dataclasses import dataclass
from enum import Enum

class EffectPolicy(Enum):
    READ_ONLY = "read_only"
    CONDITIONAL_WRITE = "conditional_write"
    WRITE = "write"

class ReviewGate(Enum):
    NONE = "none"
    CONDITIONAL = "conditional"
    MANDATORY = "mandatory"

@dataclass
class Policy:
    name: str
    trigger_event: str
    required_scopes: list
    effect_policy: EffectPolicy
    review_gate: ReviewGate
    on_approved_effects: list
    prohibited_when: list

policies = [
    Policy("合同变更审批", "合同变更申请",
           ["lease:write", "asset:read"],
           EffectPolicy.CONDITIONAL_WRITE, ReviewGate.MANDATORY,
           ["occupancy-effect", "financial-effect"], []),
    Policy("合同终止审批", "合同终止申请",
           ["lease:write"],
           EffectPolicy.CONDITIONAL_WRITE, ReviewGate.MANDATORY,
           ["state-transition-effect", "occupancy-effect", "financial-effect"],
           ["inspection未完成"]),
    Policy("费用减免审批", "费用减免申请",
           ["finance:write", "lease:read"],
           EffectPolicy.WRITE, ReviewGate.MANDATORY,
           ["financial-effect"], []),
]

for p in policies:
    print(f"Policy: {p.name}")
    print(f"  trigger: {p.trigger_event}  scopes: {p.required_scopes}")
    print(f"  effect: {p.effect_policy.value}  review: {p.review_gate.value}")
    print()

## 第 2 格：Policy 引擎——执行前门禁检查

In [ ]:
def check_policy(policy, caller_scopes, facts):
    missing = [s for s in policy.required_scopes if s not in caller_scopes]
    if missing:
        return False, f"❌ 权限不足: 缺少 {missing}"
    for pw in policy.prohibited_when:
        if facts.get(pw, False):
            return False, f"⏸️ 禁止执行: {pw}"
    if policy.review_gate == ReviewGate.MANDATORY:
        return False, "⏸️ 需要人审 (review_gate=mandatory)"
    return True, f"✅ 可执行 (effect={policy.effect_policy.value})"

agent_scopes = ["lease:write", "asset:read"]
print("Inspection未完成:", check_policy(policies[1], agent_scopes, {"inspection未完成": True}))
print("Inspection已完成:", check_policy(policies[1], agent_scopes, {"inspection未完成": False}))
print("只有read权限:", check_policy(policies[1], ["lease:read"], {}))

## 第 3 格：Ontology Compiler——编译为 Design Artifact

In [ ]:
@dataclass
class Capability:
    name: str
    description: str
    input_fields: list

@dataclass
class DesignArtifact:
    goal: str
    capability_refs: list
    trigger_event: str
    governance: dict

caps = [
    Capability("lease.terminate", "终止合同", ["contract_id", "type"]),
    Capability("lease.create", "创建合同", ["merchant_id", "resource_id"]),
]

def compile_artifact(cap_name, policy_name, caps, policies):
    cap = next(c for c in caps if c.name == cap_name)
    pol = next(p for p in policies if p.name == policy_name)
    return DesignArtifact(
        goal=f"执行 {cap.description}",
        capability_refs=[cap.name],
        trigger_event=pol.trigger_event,
        governance={
            "effect_policy": pol.effect_policy.value,
            "required_scopes": pol.required_scopes,
            "human_review_gate": pol.review_gate.value,
        }
    )

artifact = compile_artifact("lease.terminate", "合同终止审批", caps, policies)
print("Design Artifact (ADR-005):")
print(f"  goal: {artifact.goal}")
print(f"  capability: {artifact.capability_refs}")
print(f"  trigger: {artifact.trigger_event}")
print(f"  governance: {artifact.governance}")

## 第 4 格：可视化——编译链路

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_manager.fontManager.addfont("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc")
font_name = font_manager.FontProperties(fname="/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc").get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

fig, ax = plt.subplots(figsize=(10, 3))
stages = ["BCM\n审批流", "Ontology\nPolicy", "Capability\n声明", "Compiler\n编译", "Design\nArtifact"]
x = range(len(stages))
ax.plot(x, [0]*5, "o-", color="#1565C0", ms=12)
for i, s in enumerate(stages):
    ax.text(i, -0.15, s, ha="center", fontsize=9)
ax.annotate("隐式→显式", xy=(0.5, 0.05), fontsize=8, color="#F44336", ha="center")
ax.set_xlim(-0.5, 4.5); ax.set_ylim(-0.3, 0.2)
ax.set_title("Ontology Compiler 编译链路"); ax.axis("off")
plt.tight_layout()
plt.savefig("/root/learning-notebooks/第10周/d6_compiler.png", dpi=100)
plt.show()
print("编译链路图已绘制")